## Регрессия для SI

In [8]:
import numpy as np
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [9]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')

df = df.dropna().copy()

y = df['SI']

# удаляем таргет и признаки, которые создают утечку данных
X = df.drop(columns=['SI', 'IC50, mM', 'CC50, mM'])
# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#### Линейная регрессия

In [11]:
# pipeline
linreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='SI',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# обучение
linreg_pipeline.fit(X_train, y_train)

# предсказания
linreg_y_train_pred = linreg_pipeline.predict(X_train)
linreg_y_test_pred = linreg_pipeline.predict(X_test)

# метрики
linreg_train_metrics = metrics_helper.regression_metrics(y_train, linreg_y_train_pred)
linreg_test_metrics = metrics_helper.regression_metrics(y_test, linreg_y_test_pred)

print('Метрики на train:')
display(linreg_train_metrics)

print('Метрики на test:')
display(linreg_test_metrics)

Метрики на train:


{'MAE': 73.31545883904406,
 'MSE': 61983.51016490854,
 'RMSE': np.float64(248.96487737210754),
 'R2': 0.1906009619004687}

Метрики на test:


{'MAE': 4873.844159638276,
 'MSE': 4323928899.092878,
 'RMSE': np.float64(65756.58825618068),
 'R2': -2139.4277731571374}

- Линейная модель показывает низкое качество на train ($R^2 \approx 0.19$) и полностью проваливается на test ($R^2 < 0$)
- Предсказания сильно расходятся с реальными значениями

**Вывод:** линейная модель не подходит для данной задачи. Зависимость между признаками и SI носит нелинейный характер

In [12]:
# pipeline (логарифмируем таргет)
loglin_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='SI',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        TransformedTargetRegressor(
            regressor=LinearRegression(),
            func=np.log1p,
            inverse_func=np.expm1
        )
    )
])

# обучение
loglin_pipeline.fit(X_train, y_train)

# предсказания
loglin_y_train_pred = loglin_pipeline.predict(X_train)
loglin_y_test_pred = loglin_pipeline.predict(X_test)

# метрики
loglin_train_metrics = metrics_helper.regression_metrics(y_train, loglin_y_train_pred)
loglin_test_metrics = metrics_helper.regression_metrics(y_test, loglin_y_test_pred)

print('Метрики на train:')
display(loglin_train_metrics)

print('Метрики на test:')
display(loglin_test_metrics)

/Users/user/projects/Python/Compound_effectiveness_prediction/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_function_transformer.py:381: RuntimeWarning: overflow encountered in expm1
  return func(X, **(kw_args if kw_args else {}))


ValueError: Input contains infinity or a value too large for dtype('float64').

- При логарифмировании таргета модель выдает слишком большие значения, из-за чего обратное преобразование приводит к переполнению
- В предсказаниях появляются бесконечные значения, поэтому метрики посчитать невозможно

**Вывод:** логарифмирование таргета не подходит для линейной модели в задаче регрессии SI

#### Случайный лес

In [13]:
# pipeline
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='SI',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.regression_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.regression_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'MAE': 40.06078822297356,
 'MSE': 55187.229326070796,
 'RMSE': np.float64(234.91962311835678),
 'R2': 0.2793488104649381}

Метрики на test:


{'MAE': 199.9118101095854,
 'MSE': 1834701.258544588,
 'RMSE': np.float64(1354.511446442808),
 'R2': 0.0917876725357234}

- Модель показывает низкое качество на train ($R^2 \approx 0.28$) и ещё хуже на test ($R^2 \approx 0.09$)
- RandomForest не способен уловить устойчивую зависимость между признаками и таргетом
- Качество значительно хуже, чем в задачах IC50 и CC50

**Вывод:** даже нелинейная модель не справляется с задачей регрессии SI. Требуется подбор гиперпараметров

In [14]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5]
}

# GridSearch
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.regression_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.regression_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Лучшие параметры:
{'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Метрики на train:


{'MAE': 44.84541185125443,
 'MSE': 57329.137065593444,
 'RMSE': np.float64(239.43503725560603),
 'R2': 0.2513791446706787}

Метрики на test:


{'MAE': 199.41133894637503,
 'MSE': 1839025.3941913121,
 'RMSE': np.float64(1356.1067045742795),
 'R2': 0.08964714241852167}

- Подбор гиперпараметров не приводит к улучшению

**Вывод:** задача регрессии SI не решается текущим набором признаков. Таргет является производной величиной (SI = CC50 / IC50), и после исключения этих признаков модель не имеет достаточной информации для обучения